In [ ]:
!pip install requests beautifulsoup4 urllib3

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Quản lý nhà nước"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Quản lý nhà nước (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CHÍ 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí khoa học công nghệ và phát triển"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí khoa học công nghệ và phát triển (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Răng Hàm Mặt Việt Nam"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Răng Hàm Mặt Việt Nam (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học Thăng Long – Khoa học ứng dụng"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học Thăng Long – Khoa học ứng dụng (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Quy hoạch đô thị"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Quy hoạch đô thị (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CHÍ 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Y học dự phòng"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Y học dự phòng (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CHÍ MI

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Quản lý giáo dục"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Quản lý giáo dục (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CHÍ 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Tài chính - Quản trị kinh doanh"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Tài chính - Quản trị kinh doanh (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG T

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học: Toán Lý"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học: Toán Lý (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CHÍ

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học: Vật liệu và Linh kiện tiên tiến"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học: Vật liệu và Linh kiện tiên tiến (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HI

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học: Luật học"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học: Luật học (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CH

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học: Nghiên cứu Giáo dục"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học: Nghiên cứu Giáo dục (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Truyền nhiễm Việt Nam"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Truyền nhiễm Việt Nam (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Nguồn nhân lực và An sinh xã hội"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Nguồn nhân lực và An sinh xã hội (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Nghiên cứu Kinh tế và Kinh doanh châu Á"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Nghiên cứu Kinh tế và Kinh doanh châu Á (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Nghiên cứu Kinh tế"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Nghiên cứu Kinh tế (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ CÚNG HỒ CH

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học Ngôn ngữ và Văn hóa"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học Ngôn ngữ và Văn hóa (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG THỜ 

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import time
import json
import os

# Tắt cảnh báo SSL cho các site không có chứng chỉ bảo mật hợp lệ
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJOL_Scraper:
    def __init__(self, journal_code, source_name):
        self.base_url = "https://vjol.info.vn"
        self.journal_code = journal_code
        self.source_name = source_name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }
        self.results = []

    def get_direct_pdf(self, landing_url):
        """Chuyển đổi link xem PDF sang link tải xuống trực tiếp"""
        try:
            # Nếu link chứa /view/, thử đổi thành /download/ để lấy link trực tiếp
            if '/view/' in landing_url:
                download_url = landing_url.replace('/view/', '/download/')
                return download_url

            res = requests.get(landing_url, headers=self.headers, verify=False, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            # Tìm nút download hoặc iframe chứa PDF
            download_btn = soup.find('a', class_='download')
            if download_btn:
                return download_btn['href']

            iframe = soup.find('iframe', src=True)
            if iframe and '.pdf' in iframe['src'].lower():
                return iframe['src']
        except Exception as e:
            print(f"      [!] Lỗi xử lý PDF link: {e}")
        return landing_url

    def run(self):
        archive_url = f"{self.base_url}/index.php/{self.journal_code}/issue/archive"
        print(f"--- Bắt đầu quét: {self.source_name} ({self.journal_code}) ---")
        print(f"--- Link lưu trữ: {archive_url} ---")

        try:
            response = requests.get(archive_url, headers=self.headers, verify=False, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Tìm tất cả các số báo (Issues)
            issue_links = [a['href'] for a in soup.find_all('a', class_='title') if '/issue/view/' in a['href']]

            if not issue_links:
                print("[-] Không tìm thấy danh sách số báo. Kiểm tra lại journal_code.")
                return

            print(f"[*] Tìm thấy {len(issue_links)} số báo.")

            for issue_url in issue_links:
                issue_id = issue_url.split('/')[-1]
                print(f"--- Đang quét số: {issue_id} ---")

                try:
                    res_issue = requests.get(issue_url, headers=self.headers, verify=False, timeout=15)
                    soup_issue = BeautifulSoup(res_issue.text, 'html.parser')

                    # Selector cho các bài báo trong một số
                    articles = soup_issue.select('.obj_article_summary')

                    for art in articles:
                        title_tag = art.select_one('.title a')
                        # Tìm link PDF (thường có class 'pdf' hoặc 'obj_galley_link')
                        pdf_tag = art.select_one('a.obj_galley_link.pdf')

                        if title_tag and pdf_tag:
                            title = title_tag.get_text().strip()
                            pdf_landing = pdf_tag['href']

                            # Lấy link PDF cuối cùng
                            final_pdf = self.get_direct_pdf(pdf_landing)

                            self.results.append({
                                "source": self.source_name,
                                "journal_code": self.journal_code,
                                "issue_url": issue_url,
                                "article_title": title,
                                "pdf_url": final_pdf
                            })
                            print(f"    + Thu thập: {title[:50]}...")

                    # Nghỉ ngắn để tránh bị chặn
                    time.sleep(0.5)
                except Exception as e:
                    print(f"    [!] Lỗi tại số báo {issue_url}: {e}")
                    continue

            # Xuất dữ liệu ra JSON
            output_file = f"data_{self.journal_code}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)

            print("-" * 30)
            print(f"HOÀN THÀNH! Tổng cộng: {len(self.results)} bài báo.")
            print(f"Dữ liệu đã lưu tại: {os.path.abspath(output_file)}")

        except Exception as e:
            print(f"[-] Lỗi hệ thống: {e}")

if __name__ == "__main__":
    # BẠN CÓ THỂ THAY ĐỔI THÔNG TIN Ở ĐÂY
    # Ví dụ: 'vjss' cho Tạp chí Khoa học Xã hội Việt Nam
    # Hoặc 'rsr' như bài mẫu của bạn
    JOURNAL_CODE = "rsr"
    JOURNAL_NAME = "Tạp chí Khoa học và Đào tạo Thể thao"

    bot = VJOL_Scraper(JOURNAL_CODE, JOURNAL_NAME)
    bot.run()

--- Bắt đầu quét: Tạp chí Khoa học và Đào tạo Thể thao (rsr) ---
--- Link lưu trữ: https://vjol.info.vn/index.php/rsr/issue/archive ---
[*] Tìm thấy 50 số báo.
--- Đang quét số: 9812 ---
    + Thu thập: HỆ GIÁ TRỊ QUỐC GIA VIỆT NAM - LIÊN HỆ TỚI LĨNH VỰ...
    + Thu thập: CHÍNH QUYỀN CHÚA NGUYỄN VỚI SỰ HÌNH THÀNH, PHÁT TR...
    + Thu thập: TÌM HIỂU VỀ CƠ CHẾ VÀ HOẠT ĐỘNG CỦA HỘI ĐỒNG GIÁO ...
    + Thu thập: MỘT SỐ CHUYỂN BIẾN TRONG ĐỜI SỐNG TÍN NGƯỠNG CỦA C...
    + Thu thập: BIẾN ĐỔI NGHI LỄ RƯỚC NƯỚC TRONG TÍN NGƯỠNG THÀNH ...
    + Thu thập: MỘT SỐ HOẠT ĐỘNG TÔN GIÁO CỦA NHẤT QUÁN ĐẠO Ở VIỆT...
    + Thu thập: PHẬT GIÁO, ISLAM GIÁO VÀ ĐA DẠNG TÔN GIÁO Ở NAM Á ...
--- Đang quét số: 9811 ---
    + Thu thập: CÔNG NHẬN CÁC TỔ CHỨC CAO ĐÀI - DẤU ẤN PHÁP NHÂN T...
    + Thu thập: TỔ CHỨC CARITAS Ở VIỆT NAM HIỆN NAY...
    + Thu thập: HIỆN TƯỢNG THỜ HỒ CHÍ MINH Ở VIỆT NAM HIỆN NAY VÀ ...
    + Thu thập: THỜ HỒ CHÍ MINH TRONG MỐI LIÊN HỆ VỚI TRUYỀN THỐNG...
    + Thu thập: HIỆN TƯỢNG TH